In [1]:
import sys
import heapq  # Used to optimize Prim's Algorithm to O(E log V)

class DisjointSet:
    def __init__(self, vertices):
        self.parent = {v: v for v in vertices}
        self.rank = {v: 0 for v in vertices}

    def find(self, item):
        if self.parent[item] == item:
            return item
        self.parent[item] = self.find(self.parent[item])
        return self.parent[item]

    def union(self, set1, set2):
        root1 = self.find(set1)
        root2 = self.find(set2)
        if root1 != root2:
            if self.rank[root1] > self.rank[root2]:
                self.parent[root2] = root1
            else:
                self.parent[root1] = root2
                if self.rank[root1] == self.rank[root2]:
                    self.rank[root2] += 1
            return True
        return False

def prims_algorithm(vertices, edges):
    if not vertices:
        return [], 0
        
    # Build adjacency list
    adj = {v: [] for v in vertices}
    for u, v, w in edges:
        if u in adj and v in adj:  # Guard against edges referencing missing vertices
            adj[u].append((w, v))  # Store as (weight, target) for min-heap sorting
            adj[v].append((w, u))
    
    mst_edges = []
    total_cost = 0
    visited = set()
    
    # Min-heap stores elements as: (weight, current_node, parent_node)
    start_vertex = vertices[0]
    min_heap = [(0, start_vertex, None)]
    
    while min_heap and len(visited) < len(vertices):
        w, u, parent = heapq.heappop(min_heap)
        
        if u in visited:
            continue
            
        visited.add(u)
        if parent is not None:
            mst_edges.append((parent, u, w))
            total_cost += w
            
        for next_w, v in adj[u]:
            if v not in visited:
                heapq.heappush(min_heap, (next_w, v, u))
                
    # Validation check: Ensure the graph wasn't disconnected
    if len(visited) < len(vertices):
        print("\n⚠️ WARNING (Prim's): Graph is disconnected! The returned tree only covers one component.")
        
    return mst_edges, total_cost

def kruskals_algorithm(vertices, edges):
    if not vertices:
        return [], 0
        
    sorted_edges = sorted(edges, key=lambda x: x[2])
    ds = DisjointSet(vertices)
    mst_edges = []
    total_cost = 0
    
    for u, v, w in sorted_edges:
        if u in ds.parent and v in ds.parent:  # Guard against edges referencing missing vertices
            if ds.union(u, v):
                mst_edges.append((u, v, w))
                total_cost += w
                if len(mst_edges) == len(vertices) - 1:
                    break
                    
    # Validation check: An MST must always have exactly (V - 1) edges
    if len(mst_edges) < len(vertices) - 1 and len(vertices) > 1:
        print("\n⚠️ WARNING (Kruskal's): Graph is disconnected! Cannot connect all intersections.")
        
    return mst_edges, total_cost

def main():
    print("--- Infrastructure Planning Module ---")
    choice = input("Do you want to provide custom input? (yes/no): ").strip().lower()
    
    if choice == 'yes':
        try:
            vertices = input("Enter intersections separated by commas (e.g., A, B, C): ").strip().split(',')
            vertices = [v.strip() for v in vertices if v.strip()]
            
            if not vertices:
                print("Error: You must enter at least one intersection.")
                return

            num_edges = int(input("Enter number of roads: "))
            edges = []
            print("Enter each road line-by-line format -> Source, Destination, Cost")
            for i in range(num_edges):
                line = input(f"Road {i+1}: ").strip().split(',')
                if len(line) != 3:
                    print("Error: Input format invalid. Skipping this road.")
                    continue
                u, v, w = [item.strip() for item in line]
                edges.append((u, v, int(w)))
        except ValueError:
            print("Error: Invalid numeric input entered. Terminating.")
            return
    else:
        vertices = ["Intersection_A", "Intersection_B", "Intersection_C", "Intersection_D"]
        edges = [
            ("Intersection_A", "Intersection_B", 10),
            ("Intersection_A", "Intersection_C", 15),
            ("Intersection_B", "Intersection_C", 5),
            ("Intersection_B", "Intersection_D", 20),
            ("Intersection_C", "Intersection_D", 8)
        ]
        print("\nUsing Example Network Configuration:")
        for u, v, w in edges:
            print(f"  {u} <--> {v} | Cost: {w}")

    # --- Running Prim's Algorithm ---
    print("\n--- Running Prim's Algorithm ---")
    prim_mst, prim_cost = prims_algorithm(vertices, edges)
    for u, v, w in prim_mst:
        print(f"Selected Road: {u} to {v} | Cost: {w}")
    print(f"Total Infrastructure Cost (Prim's): {prim_cost}")

    # --- Running Kruskal's Algorithm ---
    print("\n--- Running Kruskal's Algorithm ---")
    kruskal_mst, kruskal_cost = kruskals_algorithm(vertices, edges)
    for u, v, w in kruskal_mst:
        print(f"Selected Road: {u} to {v} | Cost: {w}")
    print(f"Total Infrastructure Cost (Kruskal's): {kruskal_cost}")

if __name__ == "__main__":
    main()

--- Infrastructure Planning Module ---


Do you want to provide custom input? (yes/no):  no



Using Example Network Configuration:
  Intersection_A <--> Intersection_B | Cost: 10
  Intersection_A <--> Intersection_C | Cost: 15
  Intersection_B <--> Intersection_C | Cost: 5
  Intersection_B <--> Intersection_D | Cost: 20
  Intersection_C <--> Intersection_D | Cost: 8

--- Running Prim's Algorithm ---
Selected Road: Intersection_A to Intersection_B | Cost: 10
Selected Road: Intersection_B to Intersection_C | Cost: 5
Selected Road: Intersection_C to Intersection_D | Cost: 8
Total Infrastructure Cost (Prim's): 23

--- Running Kruskal's Algorithm ---
Selected Road: Intersection_B to Intersection_C | Cost: 5
Selected Road: Intersection_C to Intersection_D | Cost: 8
Selected Road: Intersection_A to Intersection_B | Cost: 10
Total Infrastructure Cost (Kruskal's): 23
